# Task 2: SVM (RBF kernel)
Dataset: Breast Cancer Wisconsin (Diagnostic), same `data.csv` used across this project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              ConfusionMatrixDisplay, classification_report)

## Load and clean the data

In [ ]:
df = pd.read_csv("data.csv")
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])

le = LabelEncoder()
df["diagnosis"] = le.fit_transform(df["diagnosis"])  # M -> 1, B -> 0

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Scale the features
Same reasoning as the linear SVM, the RBF kernel is even more sensitive to scale since it's based on distances directly.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Train the RBF SVM
Using default gamma='scale' for now, no tuning here since that's saved for the Random Forest + Optuna task.

In [ ]:
svm_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

y_pred = svm_rbf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap="Oranges")
plt.title("RBF SVM - Confusion Matrix")
plt.show()

## 2D decision boundary (via PCA)
Same PCA trick as Task 1, this time the boundary should curve since RBF isn't restricted to straight lines.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_scaled)

svm_2d = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_2d.fit(X_train_2d, y_train)

x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")
plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap="coolwarm", edgecolors="k", s=25)
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("RBF SVM decision boundary (PCA-reduced view)")
plt.show()

## Quick comparison against the linear kernel
Same test set, same split, just swapping the kernel to see if the curve actually buys us anything on this data.

In [ ]:
svm_linear_check = SVC(kernel="linear", C=1.0, random_state=42)
svm_linear_check.fit(X_train_scaled, y_train)
linear_pred = svm_linear_check.predict(X_test_scaled)

comparison = pd.DataFrame({
    "kernel": ["linear", "rbf"],
    "accuracy": [accuracy_score(y_test, linear_pred), accuracy_score(y_test, y_pred)],
    "f1_score": [f1_score(y_test, linear_pred), f1_score(y_test, y_pred)],
})
comparison